In [3]:
import numpy as np
import pandas as pd
from scipy.special import softmax
import kagglehub

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
print(f"train: {train.shape},  test: {test.shape}")


train: (2000, 8),  test: (500, 7)


In [ ]:
path = kagglehub.model_download("kamal134134/genai_project/pyTorch/deberta")
path = kagglehub.model_download("kamal134134/genai_project/pyTorch/electra")
path = kagglehub.model_download("kamal134134/genai_project/pyTorch/lstm_1")
path = kagglehub.model_download("kamal134134/genai_project/pyTorch/deberta_rag_1_4_2")

## LSTM SUBMISSION

In [8]:
OPTS = ['A', 'B', 'C', 'D', 'E']

test_deb = np.load('/kaggle/input/models/kamal134134/genai_project/pytorch/lstm_1/1/test_lstm.npy')

order = (-test_deb).argsort(axis=1)
preds = [' '.join(OPTS[k] for k in order[i, :3]) for i in range(len(test))]

sub = pd.DataFrame({'ID': test['id'].values, 'Prediction': preds})
sub.to_csv('submission_lstm.csv', index=False)
print("✓ Submission file for Deberta: submission_lstm.csv")

✓ Submission file for Deberta: submission_lstm.csv


## DEBERTA SUBMISSION

In [8]:
OPTS = ['A', 'B', 'C', 'D', 'E']

test_deb = np.load('/kaggle/input/models/kamal134134/genai_project/pytorch/deberta/3/test_deb.npy')

order = (-test_deb).argsort(axis=1)
preds = [' '.join(OPTS[k] for k in order[i, :3]) for i in range(len(test))]

sub = pd.DataFrame({'ID': test['id'].values, 'Prediction': preds})
sub.to_csv('submission_deberta.csv', index=False)
print("✓ Submission file for Deberta: submission_deberta.csv")

✓ Submission file for Deberta: submission_deberta.csv


## ELECTRA SUBMISSION

In [6]:
OPTS = ['A', 'B', 'C', 'D', 'E']

test_deb = np.load('/kaggle/input/models/kamal134134/genai_project/pytorch/electra/1/test_ele.npy')

order = (-test_deb).argsort(axis=1)
preds = [' '.join(OPTS[k] for k in order[i, :3]) for i in range(len(test))]

sub = pd.DataFrame({'ID': test['id'].values, 'Prediction': preds})
sub.to_csv('submission_electra.csv', index=False)
print("✓ Submission file for Deberta: submission_electra.csv")

✓ Submission file for Deberta: submission_electra.csv


## DEBERTA WITH RAG SUBMISSION

In [7]:
OPTS = ['A', 'B', 'C', 'D', 'E']

test_deb = np.load('/kaggle/input/models/kamal134134/genai_project/pytorch/electra/1/test_ele.npy')

order = (-test_deb).argsort(axis=1)
preds = [' '.join(OPTS[k] for k in order[i, :3]) for i in range(len(test))]

sub = pd.DataFrame({'ID': test['id'].values, 'Prediction': preds})
sub.to_csv('submission_deberta_with_Rag.csv', index=False)
print("✓ Submission file for Deberta with rag: submission_deberta_with_Rag.csv")

✓ Submission file for Deberta with rag: submission_deberta_with_Rag.csv


## DEBERTA + DEBERTA WITH RAG SUBMISSION 

In [16]:
test_csv  = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
rag_path   = '/kaggle/input/models/kamal134134/genai_project/pytorch/deberta_rag_l_4_2/2/test_deb.npy'
norag_path = '/kaggle/input/models/kamal134134/genai_project/pytorch/deberta/3/test_deb.npy'

OPTS = ['A', 'B', 'C', 'D', 'E']

W_RAG   = 0.1
W_NORAG = 0.9
T_RAG   = 1.0
T_NORAG = 0.5
POWER   = 2.0

test_df = pd.read_csv(test_csv)

logits_rag   = np.load(rag_path)
logits_norag = np.load(norag_path)

assert len(test_df) == len(logits_rag) == len(logits_norag), "Rows mismatch! Check your files."

probs_rag   = softmax(logits_rag / T_RAG, axis=1)
probs_norag = softmax(logits_norag / T_NORAG, axis=1)

blended_probs = (W_RAG * (probs_rag ** POWER)) + (W_NORAG * (probs_norag ** POWER))
blended_probs = blended_probs ** (1.0 / POWER)
blended_probs = blended_probs / np.sum(blended_probs, axis=1, keepdims=True)

predictions_strings = []
for i in range(len(blended_probs)):
    top3_idx = np.argsort(-blended_probs[i])[:3]
    top3_letters = [OPTS[idx] for idx in top3_idx]
    predictions_strings.append(" ".join(top3_letters))

sub = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions_strings
})

sub.to_csv('submission_final.csv', index=False)
print("Submission file ready.")
print(sub.head())

Submission file ready.


,id,prediction
0,1,A D E
1,2,B E A
2,3,B E D
3,4,E B A
4,5,C D A


## DEBERTA + LSTM SUBMISSION 

In [9]:
test_csv  = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
lstm_path   = '/kaggle/input/models/kamal134134/genai_project/pytorch/lstm_1/1/test_lstm.npy'
deberta_path = '/kaggle/input/models/kamal134134/genai_project/pytorch/deberta/3/test_deb.npy'

OPTS = ['A', 'B', 'C', 'D', 'E']

W_lstm = 0.1
W_deberta = 0.9
T_lstm   = 0.8
T_deberta = 1.0
POWER   = 2.0


test_df = pd.read_csv(test_csv)

logits_lstm   = np.load(lstm_path)
logits_norag = np.load(deberta_path)

probs_lstm  = softmax(logits_lstm / T_lstm, axis=1)
probs_deberta = softmax(logits_deberta / T_deberta, axis=1)

blended_probs = (W_lstm * (probs_lstm ** POWER)) + (W_deberta * (probs_deberta ** POWER))
blended_probs = blended_probs ** (1.0 / POWER)
blended_probs = blended_probs / np.sum(blended_probs, axis=1, keepdims=True)

predictions_strings = []
for i in range(len(blended_probs)):
    top3_idx = np.argsort(-blended_probs[i])[:3]
    top3_letters = [OPTS[idx] for idx in top3_idx]
    predictions_strings.append(" ".join(top3_letters))

sub = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions_strings
})

sub.to_csv('submission_final1.csv', index=False)
print("Submission file ready.")
print(sub.head())

Submission file ready.
   id prediction
0   1      A E B
1   2      B A E
2   3      B E D
3   4      E A B
4   5      C D A
